In [ ]:
## Text-based Blackjack
import random
import time
from IPython.display import clear_output

#CARD INGREDIENTS
suits =("Hearts","Diamonds","Clubs","Spades")
ranks = ("Two", "Three", "Four", "Five", "Six", "Seven", "Eight", "Nine",
         "Ten", "Jack", "King", "Queen", "Ace")
values = {"Two":2, "Three":3, "Four":4, "Five":5, "Six":6, "Seven":7, "Eight":8, 
          "Nine":9, "Ten":10,"Jack":10, "King":10, "Queen":10, "Ace": 11}

#CLASSES

#Card blueprint
class Card:
    def __init__(self, suit, rank):
        self.suit = suit
        self.rank = rank
        self.value = values[rank] 
    
    def __str__(self):
        return f"{self.rank} of {self.suit}"
        
#Deck blueprint
class Deck:
    def __init__(self):
        self.card_deck = []
        for suit in suits:
            for rank in ranks:
                self.card_deck.append(Card(suit, rank))
    
    def shuffle(self):
        random.shuffle(self.card_deck)

    def deal_card(self):
        return self.card_deck.pop()

#Hand blueprint
class Hand:
    def __init__(self):
        self.cards = []
        self.cards_total = 0
    
    def receive_card(self, card):
        self.cards.append(card)

    def hand_total(self):
        cards_total = 0
        counter = 0
        for card in self.cards:
            cards_total += card.value
            if card.rank == "Ace":
                counter += 1
        #Ace can be either 1 or 11 
        while cards_total > 21 and counter > 0:
            cards_total -= 10
            counter -= 1 
        return cards_total
    
    def __str__(self):
        return ", ".join(str(card) for card in self.cards)
    
#FUNCTIONS
def clear_screen():
    clear_output(wait=True)
    
#Play a round    
def game_on_choice():
    while True:
        print(f"Your chip total: {chip_total}")
        ans = input("Would you like to play another round (y/n)?: ")
        if ans == "y":
            #input("Press Enter to continue")
            clear_screen()
            return True
        elif ans == "n":
            print(f"Thanks for playing! Your chip total: {chip_total}")
            clear_screen()
            return False
        else:
            print("Invalid entry. Please enter 'y' or 'n'.")

#Ensure user enters a valid chip total
def chip_check():
    while True:
        try:
            chips = int(input("Please enter your chip total (value): "))
            if chips < 0:
                print("Invalid entry. Number must be greater than 0.")
            return chips
        except:
            print("Invalid entry. Please enter a whole number greater than 0.")

#Ensure bet is <= chip total                                         
def bet_check(chip_total):
    while True:
        try:
            bet = int(input("How much would you like to bet?: "))
            if bet > chip_total:
                print(f"Insufficient chips. Please place another bet. Your chip total is {chip_total}")
            elif bet < 0:
                print("Invalid entry. Bet must be greater than 0.")
            else:
                return bet
        except ValueError:
            print("Invalid entry. Please enter a whole number.")

#Check for a blackjack after intial deal
def blackjack_check(player_hand, dealer_hand):
    pl_hand_total = player_hand.hand_total()
    dea_hand_total = dealer_hand.hand_total()
    if pl_hand_total == 21 and dea_hand_total != 21:
        return "player blackjack"
    elif dea_hand_total == 21 and pl_hand_total != 21:
        return "dealer blackjack"
    elif dea_hand_total == 21 and pl_hand_total == 21:
        return "standoff"

#What happens if user hits:
def player_hit(the_deck, player_hand, dealer_hand):
    choice = "h"
    while choice == "h":
        choice = input("Would you like to hit or stand? h/s: ")
        if choice == "h":
            card = the_deck.deal_card()
            player_hand.receive_card(card)
            p_hand_total = player_hand.hand_total()
            print("Player hits...")
            print(f"Dealer Hand: {str(dealer_hand.cards[0])}, (Hidden Card)")
            time.sleep(1.5)
            print(f"Player hand: {player_hand}")
    
            if p_hand_total == 21:
                return "stand"
            elif p_hand_total > 21:
                return "player bust"
        
    if choice == "s":
        return "stand"

    else:
        return "invalid"

#What happens if player stands:
def player_stand(the_deck, player_hand, dealer_hand):
    #Display the hidden card
    print(f"Dealer hand: {dealer_hand}")
    time.sleep(1)
    dea_hand_total = dealer_hand.hand_total()
    #Dealer keeps taking cards while hand total <= 16
    while dea_hand_total <= 16:
        print("Dealer hits...")
        time.sleep(1.5)
        card = the_deck.deal_card()
        dealer_hand.receive_card(card)
        print(f"Dealer hand: {dealer_hand}")
        dea_hand_total = dealer_hand.hand_total()

    #Dealer bust
    if dea_hand_total > 21:
        return "dealer bust"
    #Else, Compare hands for results
    else:
        pl_hand_total = player_hand.hand_total()
        if dea_hand_total == pl_hand_total:
            return "tie"
        elif dea_hand_total > pl_hand_total:
            return "dealer wins"
        else:
            return "player wins"
                 
#GAME LOGIC
main_loop = True
while main_loop:
    print("WELCOME TO BLACKJACK!")
    #Enter chip total
    chip_total = chip_check()
    game_on = True
    while game_on and chip_total > 0:
        print(f"Your chip total: {chip_total}")
        #Enter bet
        bet = bet_check(chip_total)
        #Subtract bet from chip total
        chip_total -= bet
        #Initialize and Shuffle Deck
        the_deck = Deck()
        the_deck.shuffle()
        #Initialize Hands
        player_hand = Hand()
        dealer_hand = Hand()
        #Initial Deal
        card = the_deck.deal_card()
        player_hand.receive_card(card)
        card = the_deck.deal_card()
        dealer_hand.receive_card(card)
        card = the_deck.deal_card()
        player_hand.receive_card(card)
        card = the_deck.deal_card()
        dealer_hand.receive_card(card)
        #Display Hands
        print(f"Dealer Hand: {str(dealer_hand.cards[0])}, (Hidden Card)")
        print(f"Player Hand: {player_hand}")
        #Blackjack check
        res = blackjack_check(player_hand, dealer_hand)
        if res == "player blackjack":
            chip_total += 2.5*bet
            print("Player Blackjack!! Congratulations!!")
            game_on = game_on_choice()
            continue
        elif res == "dealer blackjack":
            print("Dealer Blackjack!!")
            game_on = game_on_choice()
            continue
        elif res == "standoff":
            chip_total += bet
            print("Standoff")
            game_on = game_on_choice()
            continue
        #Player Turn
        res = player_hit(the_deck, player_hand, dealer_hand)
        while res == "invalid":
            print("Invalid entry. Please enter 'h' or 's'")
            res = player_hit(the_deck, player_hand, dealer_hand)
        if res == "player bust":
            print("Player busted. Dealer wins.")
            game_on = game_on_choice()
            continue
        #Dealer Turn
        elif res == "stand":
            res = player_stand(the_deck, player_hand, dealer_hand)
            if res == "dealer bust":
                chip_total += 2*bet
                print("Dealer busted. Player wins!")
                game_on = game_on_choice()
                continue
            elif res == "dealer wins":
                print("Dealer wins!")
                game_on = game_on_choice()
                continue
            elif res == "player wins":
                chip_total += 2*bet
                print("Player wins! Congratulations!")
                game_on = game_on_choice()
                continue
            elif res == "tie":
                chip_total += bet
                print("Tie!")
                game_on = game_on_choice()
                continue
        if chip_total <= 0:
            print("Sorry, you're out of chips!")
            break

WELCOME TO BLACKJACK!


Please enter your chip total (value):  200


Your chip total: 200


How much would you like to bet?:  100


Dealer Hand: Seven of Clubs, (Hidden Card)
Player Hand: Jack of Diamonds, Ten of Diamonds


Would you like to hit or stand? h/s:  s


Dealer hand: Seven of Clubs, Six of Clubs
Dealer hits...
Dealer hand: Seven of Clubs, Six of Clubs, King of Hearts
Dealer busted. Player wins!
Your chip total: 300
